# 07 — Grounded QA Evaluation

Evaluate generated answers with two complementary layers.

**Deterministic metrics**
- Citation validity
- Evidence precision / recall / F1
- Retrieval evidence recall
- QA latency, tokens and cost

**LLM-as-a-Judge**
- Correctness
- Relevance
- Completeness
- Groundedness
- Instruction following
- Safety

Judge overhead is reported separately from production QA latency.

In [ ]:
from pathlib import Path
import sys
import os
import json

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.evaluation import (
    EvaluationDataset,
    QAEvaluator,
    QAJudge,
    format_qa_summary,
)
from src.rag.generation import OpenAIJSONGenerator
from src.rag.pipeline import GroundedQAPipeline
from src.rag.runtime import load_retrieval_stack

load_dotenv(PROJECT_ROOT / ".env")

In [ ]:
qa_config = load_config(
    PROJECT_ROOT / "configs" / "qa.yaml"
)

eval_config = load_config(
    PROJECT_ROOT / "configs" / "qa_evaluation.yaml"
)

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "retrieval_queries.json"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "qa_eval_results.jsonl"
)

SUMMARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "qa_eval_summary.json"
)

RESULTS_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "qa_eval_results.csv"
)

## Build production QA stack

In [ ]:
stack = load_retrieval_stack(PROJECT_ROOT)

qa_generator = OpenAIJSONGenerator(
    api_key=os.getenv("METIS_API_KEY"),
    base_url=os.getenv("METIS_BASE_URL"),
    model=qa_config["generation"]["model"],
    input_cost_per_million=qa_config[
        "generation"
    ].get("input_cost_per_million"),
    output_cost_per_million=qa_config[
        "generation"
    ].get("output_cost_per_million"),
)

qa_pipeline = GroundedQAPipeline(
    retriever=stack.hybrid,
    generator=qa_generator,
    documents=stack.documents,
    top_k=qa_config["qa"]["top_k"],
    max_context_chars=qa_config[
        "qa"
    ]["max_context_chars"],
    max_chars_per_comment=qa_config[
        "qa"
    ]["max_chars_per_comment"],
)

## Create LLM-as-a-Judge

In [ ]:
judge_config = eval_config["judge"]

judge_generator = OpenAIJSONGenerator(
    api_key=os.getenv("METIS_API_KEY"),
    base_url=os.getenv("METIS_BASE_URL"),
    model=judge_config["model"],
    input_cost_per_million=judge_config.get(
        "input_cost_per_million"
    ),
    output_cost_per_million=judge_config.get(
        "output_cost_per_million"
    ),
)

judge = QAJudge(
    generator=judge_generator,
    max_context_chars=judge_config["max_context_chars"],
    max_chars_per_comment=judge_config[
        "max_chars_per_comment"
    ],
)

dataset = EvaluationDataset(DATASET_PATH)

evaluator = QAEvaluator(
    qa_pipeline=qa_pipeline,
    judge=judge,
    dataset=dataset,
    weights=eval_config["weights"],
)

print("Evaluation samples available:", len(dataset))

## Run evaluation

In [ ]:
run_config = eval_config["evaluation"]

summary, results = evaluator.evaluate(
    output_path=CHECKPOINT_PATH,
    max_samples=run_config["sample_size"],
    one_query_per_product=run_config[
        "one_query_per_product"
    ],
    random_state=run_config["random_state"],
    resume=run_config["resume"],
)

print(format_qa_summary(summary))

## Save report artifacts

In [ ]:
SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

results.to_csv(
    RESULTS_CSV_PATH,
    index=False,
)

print("Summary:", SUMMARY_PATH)
print("Per-query CSV:", RESULTS_CSV_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

## Judge score breakdown

In [ ]:
score_columns = [
    "correctness",
    "relevance",
    "completeness",
    "groundedness",
    "instruction_following",
    "safety",
]

judge_scores = (
    results[score_columns]
    .mean()
    .sort_values(ascending=False)
)

display(judge_scores.to_frame("mean_score"))

ax = judge_scores.plot(
    kind="bar",
    ylim=(1, 5),
    title="LLM-as-a-Judge Scores",
)

ax.set_ylabel("Score (1-5)")
plt.tight_layout()
plt.show()

## Failure analysis

In [ ]:
display(
    evaluator.failure_cases(
        results,
        n=10,
    )
)

failure_counts = (
    pd.Series(
        [
            tag
            for tags in results["failure_tags"]
            for tag in tags
        ]
    )
    .value_counts()
)

display(
    failure_counts.to_frame("count")
)